# V3000 Molfile Guide

This notebook explains the **V3000 molfile** format for a single molecule, carefully describing the main entries and giving a worked example.

## Scope
This guide explains the V3000 *molfile* format used for a single molecule (the V3000 CTAB / connection table).
It does not try to cover V3000 rxnfiles, rgfiles, or every niche vendor extension. The goal is to explain the
core structure carefully, show what each entry means, and give a fully worked example.

## Why V3000 exists
V3000 is the newer CTfile style. Compared with V2000, it:
- removes many fixed-width limitations,
- supports larger structures,
- supports more atom and bond attributes,
- supports enhanced stereochemistry and more advanced objects.

## General structure of a V3000 molfile
A typical V3000 molfile looks like this:

```text
line 1   molecule name
line 2   program / metadata line (often writer-specific)
line 3   comment line
line 4   V3000 marker line
         M  V30 BEGIN CTAB
         M  V30 COUNTS ...
         M  V30 BEGIN ATOM
         M  V30 ... atom lines ...
         M  V30 END ATOM
         M  V30 BEGIN BOND
         M  V30 ... bond lines ...
         M  V30 END BOND
         [optional blocks]
         M  V30 END CTAB
         M  END
```

The core rule is:
- COUNTS is required.
- ATOM block is required.
- BOND block is normally present if the molecule has bonds.
- Optional blocks can follow for Sgroups, link nodes, collections, etc.

## 1. HEADER LINES

Line 1: molecule name
Example:
```text
L-Alanine
```

Meaning:
- Free-text title of the molecule.
- It is metadata only.
- It does not define chemistry.

Line 2: program / source / metadata line
Example:
```text
GSMACS-1107189510252D 1 0.00366 0.00000 0
```

Meaning:
- Historically used for program name, initials, date/time, dimensional code, scaling, energy, registry number.
- In practice, different programs write this differently.
- Readers usually do not rely on it to define bonding.
- A blank line is allowed.

Line 3: comment line
Example:
```text
Figure 1, J. Chem. Inf. Comput. Sci., Vol 32, No. 3, 1992
```

Meaning:
- Free-text comment.
- Optional metadata.
- A blank line is allowed.

Line 4: V3000 marker line
Example:
```text
0  0  0  0  999 V3000
```

Meaning:
- This marks the file as a V3000 molfile.
- The "999" is a format marker here; the real atom/bond counts are given later in the COUNTS line.
- The first three header lines are kept mainly for compatibility with older software conventions.

## 2. CTAB BLOCK

Start of CTAB
```text
M  V30 BEGIN CTAB
```

Meaning:
- Starts the V3000 connection table.
- The CTAB contains the actual chemical graph and related properties.

End of CTAB
```text
M  V30 END CTAB
```

Meaning:
- Ends the connection table.

File terminator
```text
M  END
```

Meaning:
- Ends the molfile.

## 3. COUNTS LINE

Syntax
```text
M  V30 COUNTS na nb nsg n3d chiral [REGNO=regno]
```

Fields
na
- Number of atoms.

nb
- Number of bonds.

nsg
- Number of Sgroups.

n3d
- Number of 3D constraints / 3D objects.

chiral
- 1 if the molecule is marked as chiral.
- 0 otherwise.
- This is a molecule-level flag, not the same thing as saying "there is exactly one stereocenter".

REGNO=regno
- Optional registration number.
- Used when a large registration number must be stored at CTAB level.

Example
```text
M  V30 COUNTS 6 5 0 0 1
```

Meaning:
- 6 atoms
- 5 bonds
- 0 Sgroups
- 0 3D constraints
- chiral flag set to 1

## 4. ATOM BLOCK

Block structure
```text
M  V30 BEGIN ATOM
M  V30 ... atom line(s) ...
M  V30 END ATOM
```

General atom-line syntax
```text
M  V30 index type x y z aamap [optional keyword=value fields]
```

Required atom-line fields
index
- Atom index.
- Positive integer.
- Must be unique within the molecule.
- Bond lines refer to atoms using these indices.

type
- Atom symbol or atom type.
- Common examples: C, N, O, Cl, Br.
- Query/reserved atom types also exist, such as R#, A, Q, *.
- Atom lists can also be used in query contexts.

x y z
- Cartesian coordinates of the atom.
- Usually 2D drawings have z = 0.
- 3D structures may use nonzero z coordinates.

aamap
- Atom-atom mapping number.
- Mostly used in reactions.
- 0 means "no mapping".

Optional atom keyword fields
Below are the most important V3000 atom keywords found in ordinary molfiles.

CHG=val
- Formal charge.
- Example: CHG=1, CHG=-1.
- Default is 0 (uncharged).

RAD=val
- Radical state.
- 0 = none
- 1 = singlet
- 2 = doublet
- 3 = triplet

CFG=val
- Tetrahedral stereo configuration / parity.
- 0 = none
- 1 = odd parity
- 2 = even parity
- 3 = either parity
- Important: this does NOT directly store the letters R or S.
```text
It stores local parity. Software derives R/S later using the atom arrangement plus CIP rules.
```

MASS=val
- Isotopic mass number / absolute atomic weight setting for the atom.
- Example: MASS=13 on carbon means carbon-13.

VAL=val
- Valence override.
- Used when the valence needs to be stated explicitly.
- 0 means none/default; -1 is used for zero valence in the CTfile convention.

HCOUNT=val
- Query hydrogen count.
- Mostly relevant in query molfiles, not ordinary structure storage.

STBOX=val
- Stereo-care flag used in stereochemical searching.
- Used mainly in query/search contexts.

INVRET=val
- Inversion/retention flag.
- Used in reaction contexts.

EXACHG=val
- Exact change flag.
- Reaction/query-related.

SUBST=val
- Query substitution count.

UNSAT=val
- Query unsaturation property.

RBCNT=val
- Query ring-bond-count property.

ATTCHPT=val
- Attachment point information, especially relevant for R-group definitions.

RGROUPS=(nvals ...)
- Which R-groups a given R# atom can represent.

ATTCHORD=(nvals nbr1 val1 ...)
- Attachment ordering information.

CLASS=template_class
- Template-related classification.

SEQID=sequence_id
SEQNAME=name
- Polymer / biopolymer sequence-related fields.

Practical note
In many ordinary small-molecule molfiles, an atom line uses only:
- index
- type
- x y z
- aamap
and sometimes one or more of:
- CHG
- MASS
- CFG

Example atom lines
```text
M  V30 1 C -0.6622 0.5342 0 0 CFG=2
M  V30 4 N -1.8622 -0.3695 0 0 CHG=1
M  V30 6 O 1.9464 0.4244 0 0 CHG=-1
```

How to read them:
- Atom 1 is carbon at (-0.6622, 0.5342, 0), unmapped, with tetrahedral parity 2.
- Atom 4 is nitrogen with formal charge +1.
- Atom 6 is oxygen with formal charge -1.

## 5. BOND BLOCK

Block structure
```text
M  V30 BEGIN BOND
M  V30 ... bond line(s) ...
M  V30 END BOND
```

General bond-line syntax
```text
M  V30 index type atom1 atom2 [optional keyword=value fields]
```

Required bond-line fields
index
- Bond index.
- Positive integer.
- Must be unique within the molecule.

type
- Bond type / bond order code.
- Important values:
```text
1 = single
2 = double
3 = triple
4 = aromatic
5 = single or double (query)
6 = single or aromatic (query)
7 = double or aromatic (query)
8 = any (query)
9 = coordination
10 = hydrogen bond
```

atom1 atom2
- The indices of the two atoms connected by the bond.

Optional bond keyword fields
CFG=val
- Bond stereo / wedge-like bond direction.
- 0 = none
- 1 = up
- 2 = either
- 3 = down
- Often used together with atom stereo to define tetrahedral stereochemistry from a 2D drawing.

TOPO=val
- Query topology.
- 0 = not specified
- 1 = ring
- 2 = chain

RXCTR=val
- Reaction center status.
- Used in reaction files.

STBOX=val
- Stereo-care box for stereo searching.

ATTACH=[ALL|ANY]
ENDPTS=(natoms atom1 atom2 ...)
- Multiple-endpoint / organometallic style bond representation.
- Used for special cases where one connection is conceptually attached to multiple endpoints.

DISP=[HBOND1|HBOND2|COORD|DATIVE]
- Display style for some special bond types.
- COORD and DATIVE are relevant to coordination/dative bonds.
- HBOND1 and HBOND2 are hydrogen-bond display styles.

Example bond lines
```text
M  V30 1 1 1 2
M  V30 2 1 1 3 CFG=1
M  V30 4 2 2 5
```

How to read them:
- Bond 1 is a single bond between atom 1 and atom 2.
- Bond 2 is a single bond between atom 1 and atom 3, with bond stereo CFG=1 (up).
- Bond 4 is a double bond between atom 2 and atom 5.

## 6. LINKNODE LINE

Syntax
```text
M  V30 LINKNODE minrep maxrep nbonds inatom outatom [inatom outatom ...]
```

Meaning
- Describes repeatable connection patterns.
- Mostly used for repeating groups / special representations.
- Not common in ordinary small-molecule files.

Fields
minrep
- Minimum number of repetitions.
- In the published CTfile description, this is fixed as 1 for future expansion and not currently used.

maxrep
- Maximum number of repetitions.

nbonds
- Number of directed bonds defining the repeating group.

inatom
- Atom inside the repeating group.

outatom
- Atom outside the repeating group but bonded to the inatom.

## 7. SGROUP BLOCK

Why Sgroups exist
Sgroups store grouped structural or annotation information, for example:
- abbreviations / superatoms,
- polymer repeating units,
- data fields attached to parts of a structure,
- generic and Markush-style information.

Block structure
```text
M  V30 BEGIN SGROUP
M  V30 ... Sgroup entries ...
M  V30 END SGROUP
```

General Sgroup syntax
```text
M  V30 index type extindex [keyword fields]
```

Important Sgroup fields
index
- Sgroup index.

type
- Sgroup type.
- Important examples:
```text
SUP = superatom / abbreviation
MUL = multiple group
SRU = structural repeating unit
MON = monomer
COP = copolymer
CRO = crosslink
MOD = modification
GRA = graft
COM = component
MIX = mixture
FOR = formulation
DAT = data Sgroup
ANY = generic
GEN = generic
```

extindex
- External index value.

ATOMS=(natoms ...)
- Atoms that belong to the Sgroup.

XBONDS=(nxbonds ...)
- Crossing bonds of the Sgroup.

CBONDS=(ncbonds ...)
- Containment bonds; used especially for data Sgroups.

PATOMS=(npatom ...)
- Paradigmatic repeating unit atoms for multiple groups.

SUBTYPE=subtype
- Sgroup subtype, for example ALT, RAN, BLO for polymer-style specification.

MULT=mult
- Multiplier for multiple groups.

CONNECT=EU|HH|HT
- Connectivity pattern for repeating units.
- EU = either/unknown (default)
- HH = head-to-head
- HT = head-to-tail

PARENT=parent
- Parent Sgroup index.

COMPNO=compno
- Component order number.

XBHEAD=(...)
XBCORR=(...)
- Crossing-bond head assignments and correspondence information.

LABEL=label
- Display label.

BRKXYZ=(...)
BRKTYP=BRACKET|PAREN
- Bracket display coordinates and bracket style.

FIELDNAME, FIELDINFO, FIELDDISP, QUERYTYPE, QUERYOP, FIELDDATA
- Data-Sgroup related metadata and values.

CLASS=class
- Sgroup class.

SAP=(...)
- Sgroup attachment point info.

SEQID, NATREPLACE, SEQNAME
- Polymer / biopolymer oriented fields.

Practical note
If your file is just a normal small organic molecule, there may be no SGROUP block at all.
In that case, the COUNTS line usually has nsg = 0.

## 8. COLLECTION BLOCK

Why it exists
The collection block stores grouped metadata such as enhanced stereochemistry sets.

Typical entries include:
- MDLV30/STEABS  -> absolute stereochemical group
- MDLV30/STEREL  -> relative (OR) stereochemical group
- MDLV30/STERAC  -> racemic / AND stereochemical group

These matter when a file contains multiple stereocenters and the file needs to specify whether they are:
- absolute,
- relative,
- racemic mixtures,
- or grouped in a particular stereo relationship.

This block is optional.

## 9. WHAT V3000 DOES *NOT* DO DIRECTLY

Literal R/S labels
V3000 does not normally store the literal letters R or S on an atom line.
Instead, tetrahedral stereochemistry is encoded through local parity information such as CFG on the atom,
plus bond direction or coordinates. Software can then derive R or S by applying the CIP priority rules.

Literal E/Z labels
Similarly, double-bond stereochemistry is not usually stored as the literal letters E or Z.
It is represented through geometry / bond stereo conventions, from which software can determine E or Z.

## 10. FULL WORKED EXAMPLE

Example file
```text
L-Alanine
GSMACS-1107189510252D 1 0.00366 0.00000 0
Figure 1, J. Chem. Inf. Comput. Sci., Vol 32, No. 3, 1992
0  0  0  0  999 V3000
M  V30 BEGIN CTAB
M  V30 COUNTS 6 5 0 0 1
M  V30 BEGIN ATOM
M  V30 1 C -0.6622 0.5342 0 0 CFG=2
M  V30 2 C 0.6622 -0.3000 0 0
M  V30 3 O -0.7207 2.0817 0 0 MASS=13
M  V30 4 N -1.8622 -0.3695 0 0 CHG=1
M  V30 5 O 0.6220 -1.8037 0 0
M  V30 6 O 1.9464 0.4244 0 0 CHG=-1
M  V30 END ATOM
M  V30 BEGIN BOND
M  V30 1 1 1 2
M  V30 2 1 1 3 CFG=1
M  V30 3 1 1 4
M  V30 4 2 2 5
M  V30 5 1 2 6
M  V30 END BOND
M  V30 END CTAB
M  END
```

Annotated explanation
1) L-Alanine
- Molecule name.

2) GSMACS-1107189510252D 1 0.00366 0.00000 0
- Writer/program metadata line.

3) Figure 1, J. Chem. Inf. Comput. Sci., Vol 32, No. 3, 1992
- Comment line.

4) 0  0  0  0  999 V3000
- V3000 marker line.

5) M  V30 BEGIN CTAB
- Start of connection table.

6) M  V30 COUNTS 6 5 0 0 1
- 6 atoms, 5 bonds, 0 Sgroups, 0 3D constraints, chiral flag set.

7) M  V30 BEGIN ATOM
- Start of atom block.

8) M  V30 1 C -0.6622 0.5342 0 0 CFG=2
- Atom 1: carbon, at the given coordinates, no mapping, tetrahedral parity 2.

9) M  V30 2 C 0.6622 -0.3000 0 0
- Atom 2: carbon.

10) M  V30 3 O -0.7207 2.0817 0 0 MASS=13
- Atom 3: oxygen with isotope/mass specification 13.

11) M  V30 4 N -1.8622 -0.3695 0 0 CHG=1
- Atom 4: positively charged nitrogen.

12) M  V30 5 O 0.6220 -1.8037 0 0
- Atom 5: oxygen.

13) M  V30 6 O 1.9464 0.4244 0 0 CHG=-1
- Atom 6: negatively charged oxygen.

14) M  V30 END ATOM
- End of atom block.

15) M  V30 BEGIN BOND
- Start of bond block.

16) M  V30 1 1 1 2
- Bond 1: single bond between atoms 1 and 2.

17) M  V30 2 1 1 3 CFG=1
- Bond 2: single bond between atoms 1 and 3, bond stereo flag CFG=1 (up).

18) M  V30 3 1 1 4
- Bond 3: single bond between atoms 1 and 4.

19) M  V30 4 2 2 5
- Bond 4: double bond between atoms 2 and 5.

20) M  V30 5 1 2 6
- Bond 5: single bond between atoms 2 and 6.

21) M  V30 END BOND
- End of bond block.

22) M  V30 END CTAB
- End of connection table.

23) M  END
- End of file.

What chemical pattern this example represents
From the connectivity:
- atom 1 is bonded to atoms 2, 3, and 4,
- atom 2 is bonded to atoms 1, 5, and 6,
- bond 2-5 is double,
- atom 4 has charge +1,
- atom 6 has charge -1.

So this is an amino-acid-like zwitterionic skeleton with explicit stereochemical information.

## 11. QUICK READING CHECKLIST

When you open a V3000 molfile, read it in this order:

## 1. Check line 4 for "V3000".

## 2. Find BEGIN CTAB.

## 3. Read the COUNTS line.

## 4. Read the ATOM block:
```text
 - index
 - element/type
 - coordinates
 - CHG / MASS / CFG if present
```

## 5. Read the BOND block:
```text
 - bond type
 - atom indices connected
 - bond CFG if present
```

## 6. Check whether optional blocks exist:
```text
 - SGROUP
 - LINKNODE
 - COLLECTION
```

## 7. End at M  END.

## 12. COMMON PITFALLS

## 1. Confusing atom index with bond index
- Atom and bond indices are separate numbering systems.

## 2. Thinking CFG directly equals R/S
- It does not.
- CFG is local parity/direction information.
- R/S must be derived later.

## 3. Thinking the chiral flag means "one stereocenter"
- It does not.
- It is a molecule-level flag.

## 4. Thinking line 2 is chemically essential
- Usually it is not.
- It is mostly metadata.

## 5. Forgetting that many advanced keywords are query/reaction-specific
- Ordinary structure files often use only a small subset of V3000.

## 13. MINIMAL TEMPLATE YOU CAN REUSE

```text
MoleculeName
ProgramOrBlank
CommentOrBlank
0  0  0  0  999 V3000
M  V30 BEGIN CTAB
M  V30 COUNTS <na> <nb> <nsg> <n3d> <chiral>
M  V30 BEGIN ATOM
M  V30 <atom_index> <atom_type> <x> <y> <z> <aamap> [options]
...
M  V30 END ATOM
M  V30 BEGIN BOND
M  V30 <bond_index> <bond_type> <atom1> <atom2> [options]
...
M  V30 END BOND
[optional blocks]
M  V30 END CTAB
M  END
```

## 14. REFERENCES USED TO PREPARE THIS GUIDE

Primary references consulted:
- BIOVIA Databases 2020, CTFile Formats (official CTfile specification).
- ChemAxon documentation on MDL MOL / SD / RXN / RDF formats and V3000-supported atom/bond fields.

End of guide.

## Reusable example

The example below is included as a Python string so you can copy it directly into parsing or testing code.

In [ ]:
v3000_example = """L-Alanine
GSMACS-1107189510252D 1 0.00366 0.00000 0
Figure 1, J. Chem. Inf. Comput. Sci., Vol 32, No. 3, 1992
  0  0  0  0  999 V3000
M  V30 BEGIN CTAB
M  V30 COUNTS 6 5 0 0 1
M  V30 BEGIN ATOM
M  V30 1 C -0.6622 0.5342 0 0 CFG=2
M  V30 2 C 0.6622 -0.3 0 0
M  V30 3 O -0.7207 2.0817 0 0 MASS=13
M  V30 4 N -1.8622 -0.3695 0 0 CHG=1
M  V30 5 O 0.622 -1.8037 0 0
M  V30 6 O 1.9464 0.4244 0 0 CHG=-1
M  V30 END ATOM
M  V30 BEGIN BOND
M  V30 1 1 1 2
M  V30 2 1 1 3 CFG=1
M  V30 3 1 1 4
M  V30 4 2 2 5
M  V30 5 1 2 6
M  V30 END BOND
M  V30 END CTAB
M  END
"""

print(v3000_example)